# Invoice Anomaly Analysis

Identify unusual vendor-invoice transactions using unsupervised anomaly detection. This notebook deliberately avoids presenting the rule-derived `flagged_invoice` label as ML ground truth.

**Method:** Isolation Forest  
**Preprocessing:** RobustScaler  
**Goal:** rank unusual invoice transactions for investigation, not automatically classify fraud.

## 1. Setup
Place `data.db` at the repository root before running the notebook.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.data import load_table
from src.invoice_anomaly import build_detector, FEATURES

In [ ]:
df = load_table('vendor_invoice').copy()
df['PODate'] = pd.to_datetime(df['PODate'], errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['PayDate'] = pd.to_datetime(df['PayDate'], errors='coerce')
df['days_po_to_invoice'] = (df['InvoiceDate'] - df['PODate']).dt.days
df['days_to_pay'] = (df['PayDate'] - df['InvoiceDate']).dt.days
df['total_brands'] = 1
df['total_quantity'] = df['Quantity']
df['total_dollars'] = df['Dollars']
df['avg_receiving_delay'] = 0
df = df.dropna(subset=['Dollars', 'Quantity', 'Freight']).reset_index(drop=True)
df.shape

## 2. Inspect transaction features

In [ ]:
df[FEATURES].describe().round(2)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Dollars'], df['Freight'], alpha=0.4)
plt.xlabel('Invoice Dollars')
plt.ylabel('Freight Cost')
plt.title('Invoice Value vs Freight Cost')
plt.show()

## 3. Fit Isolation Forest
Isolation Forest is used because there is no independently verified fraud/anomaly target. The model learns patterns in the transaction features and assigns an anomaly score.

In [ ]:
detector = build_detector()
detector.fit(df[FEATURES])
df['is_anomaly'] = detector.predict(df[FEATURES]) == -1
df['anomaly_score'] = -detector.decision_function(df[FEATURES])
print(f'Rows scored: {len(df):,}')
print(f'Anomalies flagged: {df.is_anomaly.sum():,}')
print(f'Observed anomaly rate: {df.is_anomaly.mean():.2%}')

## 4. Highest-risk transactions
Higher anomaly scores indicate observations that are more unusual relative to the fitted data distribution.

In [ ]:
top_anomalies = df.sort_values('anomaly_score', ascending=False)
top_anomalies[['PONumber', 'VendorNumber', 'Dollars', 'Quantity', 'Freight', 'days_po_to_invoice', 'days_to_pay', 'anomaly_score']].head(20)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Dollars'], df['Freight'], c=df['is_anomaly'], alpha=0.45)
plt.xlabel('Invoice Dollars')
plt.ylabel('Freight Cost')
plt.title('Invoice Transactions and Detected Anomalies')
plt.show()

## 5. Interpretation
An anomaly is a transaction that looks unusual to the model; it is **not proof of fraud or an error**. In a production workflow, the ranked records would be reviewed against business rules and source documents before any action is taken.

The reusable detector is implemented in `src/invoice_anomaly.py`, and `scripts/run_anomaly_detection.py` exports scored transactions and the trained artifact.